# Supplier-share census review

**Status:** exploratory companion notebook (non-citable)  
**Research question:** F2 - supplier-track share of the SBIR/STTR portfolio  
**Canonical computation:** `scripts/data/build_supplier_share_census.py`  
**Frozen design:** `specs/supplier-share-census/design.md` and `amendments.md`  

This notebook only reviews canonical artifacts. It does not recompute firm identity, persistence, venture signals, or the sensitivity grid. A missing venture-search input remains unknown rather than becoming evidence of no venture signal.

In [ ]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import SVG, display


def find_repo_root(start: Path = Path.cwd()) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists() and (candidate / "sbir_etl").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the sbir-analytics checkout")


REPO_ROOT = find_repo_root()
REPORT_DIR = REPO_ROOT / "data" / "reports" / "supplier_share_census"
ARTIFACTS = {
    "manifest": REPORT_DIR / "supplier_share_manifest.json",
    "firm grid": REPORT_DIR / "supplier_share_firm_grid.parquet",
    "summary": REPORT_DIR / "supplier_share_summary.csv",
    "readout": REPORT_DIR / "supplier_share_readout.md",
    "cohort figure": REPORT_DIR / "analysis" / "supplier_share_cohort_curve.svg",
}
pd.DataFrame(
    [
        {"artifact": name, "path": str(path.relative_to(REPO_ROOT)), "exists": path.exists()}
        for name, path in ARTIFACTS.items()
    ]
)

## Data contract

- **Population:** every nonblank exact source-company label in the legacy M&A-study denominator, collapsed with the frozen firm-envelope policy.
- **Grain:** one pseudonymous canonical firm per sensitivity-grid cell in Parquet; aggregate matrix cell by stratum in CSV.
- **Central grid:** persistence tenure 10 years, 6 awards, and a 15-year maturity gate.
- **Headline rule:** report persistent plus no-venture-signal only when required Form D and M&A channels are searchable for every mature denominator firm.
- **Use:** descriptive census only. All outputs remain exploratory and non-citable until the frozen validation gates pass.

In [ ]:
manifest_path = ARTIFACTS["manifest"]
if not manifest_path.exists():
    raise FileNotFoundError(
        f"Missing {manifest_path.relative_to(REPO_ROOT)}; run "
        "scripts/data/build_supplier_share_census.py first."
    )
manifest = json.loads(manifest_path.read_text())
display(
    pd.DataFrame(
        {
            "value": {
                "schema_version": manifest["schema_version"],
                "epistemic_tier": manifest["epistemic_tier"],
                "citable": manifest["citable"],
                "validation_status": manifest["validation"]["status"],
                "source_company_labels": manifest["inputs"]["sbir_awards"]["source_company_labels"],
                "canonical_firms": manifest["inputs"]["sbir_awards"]["canonical_firms"],
                "form_d_available": manifest["inputs"]["form_d"]["available"],
                "ma_events_available": manifest["inputs"]["ma"]["events_available"],
                "ma_scan_available": manifest["inputs"]["ma"]["scan_available"],
                "contracts_available": manifest["inputs"]["contracts"]["available"],
            }
        }
    )
)

## Validation and suppression

The manifest is the first stop. A blocked validation status or incomplete required search channel suppresses the supplier-cell headline; it does not convert unknown firms into the no-venture cell.

In [ ]:
summary = pd.read_csv(ARTIFACTS["summary"])
firm_grid = pd.read_parquet(ARTIFACTS["firm grid"])
assert not firm_grid.duplicated(["firm_id", "t_years", "n_awards", "window_years"]).any()
assert not summary["citable"].any()
display(
    firm_grid.groupby(["validation_status", "venture_state"], dropna=False)
    .size()
    .rename("firm_grid_rows")
    .reset_index()
)

## Central 2 x 2 matrix

The four requested cells are retained alongside explicit unknown-venture cells. Agency firm counts are non-additive because a firm can receive awards from multiple agency groups; dollars remain attached to their awards.

In [ ]:
central = summary.loc[
    summary["is_central_grid"]
    & summary["stratification"].eq("overall")
    & summary["scope"].eq("headline_mature")
].copy()
central.loc[
    central["matrix_cell"].ne("TOTAL"),
    ["matrix_cell", "firm_count", "firm_share", "sbir_dollars", "dollar_share"],
].sort_values("matrix_cell")

## Frozen sensitivity grid

A headline must survive every combination of tenure threshold, award-count threshold, and maturity window. Blank supplier shares mean suppressed, not zero.

In [ ]:
grid = summary.loc[
    summary["stratification"].eq("overall")
    & summary["scope"].eq("headline_mature")
    & summary["matrix_cell"].eq("TOTAL"),
    [
        "t_years",
        "n_awards",
        "window_years",
        "total_firms",
        "measurable_firm_count",
        "headline_available",
        "supplier_firm_share",
        "supplier_dollar_share",
        "validation_status",
    ],
].sort_values(["window_years", "t_years", "n_awards"])
grid

## Agency and cohort views

These are the requested stratifications at the central grid. Cohort rows expose the maturity gradient rather than hiding it in a pooled estimate.

In [ ]:
agency = summary.loc[
    summary["is_central_grid"]
    & summary["stratification"].eq("agency")
    & summary["scope"].eq("headline_mature")
    & summary["matrix_cell"].ne("TOTAL"),
    ["stratum", "matrix_cell", "firm_count", "sbir_dollars", "dollar_share"],
]
display(agency.sort_values(["stratum", "matrix_cell"]))

cohort = summary.loc[
    summary["is_central_grid"]
    & summary["stratification"].eq("first_award_year")
    & summary["matrix_cell"].eq("TOTAL"),
    [
        "stratum",
        "scope",
        "total_firms",
        "measurable_firm_count",
        "headline_available",
        "supplier_dollar_share",
    ],
].sort_values(["stratum", "scope"])
display(cohort.tail(30))

dollar_deciles = summary.loc[
    summary["is_central_grid"]
    & summary["stratification"].eq("cumulative_dollar_decile")
    & summary["scope"].eq("headline_mature")
    & summary["matrix_cell"].ne("TOTAL"),
    ["stratum", "matrix_cell", "firm_count", "sbir_dollars", "dollar_share"],
]
dollar_deciles.sort_values(["stratum", "matrix_cell"])

In [ ]:
figure_path = ARTIFACTS["cohort figure"]
if figure_path.exists():
    display(SVG(filename=str(figure_path)))
else:
    print(f"Missing {figure_path.relative_to(REPO_ROOT)}; run the producer first.")

## Interpretation discipline

The neutral construct is sustained federal performance with no observed venture signal. The same estimate can be framed favorably as mission-supplier continuity or critically as a "mills share"; neither reading changes the deterministic definition.

The main directional limitations oppose each other. Prime-only FPDS and under-coded Phase III activity undercount federal continuation. Form D and public filing absence misses bootstrap, debt, private placements outside Form D, and revenue-funded growth, which overcounts the no-venture population. Young cohorts are right-censored, identities can remain unresolved, and dollars are nominal. Net bias is ambiguous.

Promotion requires complete required-channel search coverage, approximately 50 stratified hand reviews with cell-level agreement, face-validity anchors, and the frozen negative-control result. No per-firm names belong in public outputs.